In [1]:
# Split a large CSV into multiple files under a target size, preserving the header.

import csv
import os

In [6]:
# --- Set these two variables ---
input_path = "../results/results_df_1782793884.csv"   # path to the large CSV
max_mb = 25                         # max size per chunk, in MB (stay under the 30MB limit)

In [7]:
def split_csv(input_path, max_mb=25):
    max_bytes = max_mb * 1024 * 1024
    base, ext = os.path.splitext(input_path)

    with open(input_path, "r", newline="", encoding="utf-8") as infile:
        reader = csv.reader(infile)
        header = next(reader)

        part_num = 1
        out_path = f"{base}_part{part_num}{ext}"
        outfile = open(out_path, "w", newline="", encoding="utf-8")
        writer = csv.writer(outfile)
        writer.writerow(header)
        current_size = outfile.tell()
        written_files = [out_path]

        for row in reader:
            row_line = ",".join(row) + "\n"
            row_size = len(row_line.encode("utf-8"))

            if current_size + row_size > max_bytes:
                outfile.close()
                part_num += 1
                out_path = f"{base}_part{part_num}{ext}"
                outfile = open(out_path, "w", newline="", encoding="utf-8")
                writer = csv.writer(outfile)
                writer.writerow(header)
                current_size = outfile.tell()
                written_files.append(out_path)

            writer.writerow(row)
            current_size += row_size

        outfile.close()

    print(f"Split into {len(written_files)} file(s):")
    for f in written_files:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  {f}  ({size_mb:.2f} MB)")

    return written_files

written_files = split_csv(input_path, max_mb)

Split into 4 file(s):
  ../results/results_df_1782793884_part1.csv  (25.10 MB)
  ../results/results_df_1782793884_part2.csv  (25.10 MB)
  ../results/results_df_1782793884_part3.csv  (25.11 MB)
  ../results/results_df_1782793884_part4.csv  (19.92 MB)
